# Mercado Libre — Decision Intelligence Analysis 2024

**Project:** decision-intelligence-latam  
**Author:** Lautaro Ramos  
**Scope:** Fiscal Year 2024 | Markets: Brazil & Mexico  
**Data sources:** MELI 10-K (SEC), Q4 2024 Earnings Release, Investor Relations

---

### Analytical Framework

This notebook integrates three lenses:
1. **Financial** — Key metrics: GMV, revenue, take rate, unit economics
2. **Risk** — Mercado Crédito: portfolio quality, NPL, coverage ratios
3. **Decision Intelligence** — How data-driven decisions translate into measurable business outcomes


In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.labelsize':   10,
})

MELI_YELLOW = '#ffe600'
MELI_BLUE   = '#2d73f5'
ACCENT_RED  = '#f85149'
ACCENT_GREEN= '#3fb950'

print('Libraries loaded. Ready.')

---
## 1. Financial Overview
Source: MELI 10-K FY2024, Q4 2024 Earnings Release

In [ ]:
# ── 1.1  Annual P&L Summary (USD millions) ────────────────────────────────────
# Source: MELI 10-K FY2024
financials = pd.DataFrame({
    'Metric': [
        'Gross Merchandise Volume (GMV)',
        'Total Payment Volume (TPV)',
        'Net Revenue',
        'Commerce Revenue',
        'Fintech Revenue',
        'Gross Profit',
        'Operating Income',
        'Net Income',
    ],
    'FY2023_USD_M': [40_005, 211_576, 14_474, 7_225, 7_249, 6_467, 1_498, 1_124],
    'FY2024_USD_M': [50_490, 262_700, 20_777, 10_469, 10_308, 9_274, 2_649, 1_912],
})

financials['YoY_Growth_%'] = (
    (financials['FY2024_USD_M'] - financials['FY2023_USD_M']) / financials['FY2023_USD_M'] * 100
).round(1)

print('=== MERCADO LIBRE — FY2024 FINANCIAL SUMMARY ===')
print(financials.to_string(index=False))

In [ ]:
# ── 1.2  Revenue breakdown: Commerce vs Fintech ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Mercado Libre — Revenue Composition FY2023 vs FY2024', fontsize=14, color='#e6edf3', y=1.02)

years   = ['FY2023', 'FY2024']
commerce = [7_225,  10_469]
fintech  = [7_249,  10_308]

# Stacked bar
ax = axes[0]
bars_c = ax.bar(years, commerce, color=MELI_YELLOW, label='Commerce', width=0.45)
bars_f = ax.bar(years, fintech,  bottom=commerce, color=MELI_BLUE, label='Fintech', width=0.45)
ax.set_title('Revenue Split (USD M)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}B'))
ax.legend(facecolor='#161b22', edgecolor='#30363d')
ax.grid(axis='y')

# Fintech share evolution
ax2 = axes[1]
shares = [f/(c+f)*100 for c, f in zip(commerce, fintech)]
ax2.bar(years, shares, color=[MELI_BLUE, ACCENT_GREEN], width=0.35)
for i, (yr, sh) in enumerate(zip(years, shares)):
    ax2.text(i, sh + 0.3, f'{sh:.1f}%', ha='center', color='#e6edf3', fontsize=11)
ax2.set_title('Fintech Share of Total Revenue (%)')
ax2.set_ylim(0, 60)
ax2.grid(axis='y')

plt.tight_layout()
plt.savefig('../reports/figures/01_revenue_split.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── 1.3  GMV quarterly trend & take rate ─────────────────────────────────────
# Source: MELI quarterly earnings releases 2024
quarterly = pd.DataFrame({
    'Quarter':      ['Q1 23', 'Q2 23', 'Q3 23', 'Q4 23', 'Q1 24', 'Q2 24', 'Q3 24', 'Q4 24'],
    'GMV_USD_M':    [8_950, 9_987, 10_115, 10_953, 11_373, 12_563, 13_144, 13_410],
    'Revenue_USD_M':[3_040, 3_416, 3_760, 4_258, 4_334, 5_073, 5_309, 6_061],
})
quarterly['Take_Rate_%'] = (quarterly['Revenue_USD_M'] / quarterly['GMV_USD_M'] * 100).round(2)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.bar(quarterly['Quarter'], quarterly['GMV_USD_M'], color=MELI_YELLOW, alpha=0.75, label='GMV')
ax2.plot(quarterly['Quarter'], quarterly['Take_Rate_%'], color=ACCENT_GREEN,
         marker='o', linewidth=2.5, label='Take Rate %')

ax1.set_title('GMV vs Take Rate — Quarterly 2023–2024', fontsize=13)
ax1.set_ylabel('GMV (USD M)', color=MELI_YELLOW)
ax2.set_ylabel('Take Rate (%)', color=ACCENT_GREEN)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.1f}B'))
ax1.grid(axis='y', alpha=0.4)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.savefig('../reports/figures/02_gmv_takerate.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(quarterly[['Quarter','GMV_USD_M','Revenue_USD_M','Take_Rate_%']].to_string(index=False))

---
## 2. Risk Analysis — Mercado Crédito
Source: MELI 10-K FY2024, Supplemental Financial Data

Mercado Crédito is MELI's credit arm, offering consumer and merchant loans using proprietary behavioral data from the marketplace and payments ecosystem. Risk management here is inseparable from data strategy.

In [ ]:
# ── 2.1  Credit Portfolio Evolution ──────────────────────────────────────────
credit = pd.DataFrame({
    'Quarter':          ['Q1 23', 'Q2 23', 'Q3 23', 'Q4 23', 'Q1 24', 'Q2 24', 'Q3 24', 'Q4 24'],
    'Portfolio_USD_M':  [2_954,   3_282,   3_597,   3_762,   4_284,   4_752,   5_182,   6_577],
    'NPL_90d_%':        [8.4,     7.6,     6.8,     6.3,     6.0,     5.9,     6.1,     6.4],
    'Coverage_Ratio_%': [88,      91,      94,      96,      98,      100,     99,      97],
})

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
fig.suptitle('Mercado Crédito — Portfolio Quality 2023–2024', fontsize=14, color='#e6edf3')

# Portfolio size
axes[0].fill_between(credit['Quarter'], credit['Portfolio_USD_M'], alpha=0.3, color=MELI_BLUE)
axes[0].plot(credit['Quarter'], credit['Portfolio_USD_M'], color=MELI_BLUE, marker='o', linewidth=2.5)
axes[0].set_ylabel('Portfolio (USD M)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.1f}B'))
axes[0].grid()
for i, (q, v) in enumerate(zip(credit['Quarter'], credit['Portfolio_USD_M'])):
    if i % 2 == 0:
        axes[0].annotate(f'${v/1000:.2f}B', (q, v), textcoords='offset points', xytext=(0, 8),
                         ha='center', fontsize=8, color='#8b949e')

# NPL & Coverage
ax_npl = axes[1]
ax_cov = ax_npl.twinx()
ax_npl.plot(credit['Quarter'], credit['NPL_90d_%'], color=ACCENT_RED, marker='s', linewidth=2.5, label='NPL >90d (%)')
ax_cov.plot(credit['Quarter'], credit['Coverage_Ratio_%'], color=ACCENT_GREEN, marker='^', linewidth=2.5, label='Coverage Ratio (%)')
ax_npl.set_ylabel('NPL >90d (%)', color=ACCENT_RED)
ax_cov.set_ylabel('Coverage Ratio (%)', color=ACCENT_GREEN)
ax_npl.set_ylim(0, 15)
ax_cov.set_ylim(70, 115)
ax_npl.grid(alpha=0.4)

lines1, labels1 = ax_npl.get_legend_handles_labels()
lines2, labels2 = ax_cov.get_legend_handles_labels()
ax_npl.legend(lines1 + lines2, labels1 + labels2, facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.savefig('../reports/figures/03_credit_quality.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(credit.to_string(index=False))

In [ ]:
# ── 2.2  Risk-Adjusted Revenue Estimation ────────────────────────────────────
# Approximation: Fintech Revenue - Provision for Doubtful Accounts
# Source: MELI 10-K FY2024 Notes to Financial Statements

risk_adj = pd.DataFrame({
    'Year':                  ['FY2022', 'FY2023', 'FY2024'],
    'Fintech_Rev_USD_M':     [4_929,    7_249,    10_308],
    'Provision_USD_M':       [1_508,    1_954,    2_403],
    'Credit_Portfolio_USD_M':[2_014,    3_762,    6_577],
})

risk_adj['Risk_Adj_Rev_USD_M'] = risk_adj['Fintech_Rev_USD_M'] - risk_adj['Provision_USD_M']
risk_adj['Provision_Rate_%']   = (risk_adj['Provision_USD_M'] / risk_adj['Credit_Portfolio_USD_M'] * 100).round(1)
risk_adj['RAR_Margin_%']       = (risk_adj['Risk_Adj_Rev_USD_M'] / risk_adj['Fintech_Rev_USD_M'] * 100).round(1)

print('=== RISK-ADJUSTED FINTECH REVENUE ===')
print(risk_adj.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(risk_adj['Year']))
ax.bar(x - 0.2, risk_adj['Fintech_Rev_USD_M'], 0.35, label='Fintech Revenue', color=MELI_BLUE)
ax.bar(x + 0.2, risk_adj['Risk_Adj_Rev_USD_M'], 0.35, label='Risk-Adjusted Revenue', color=ACCENT_GREEN)
ax.set_xticks(x)
ax.set_xticklabels(risk_adj['Year'])
ax.set_title('Fintech Revenue vs Risk-Adjusted Revenue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.1f}B'))
ax.legend(facecolor='#161b22', edgecolor='#30363d')
ax.grid(axis='y')
plt.tight_layout()
plt.savefig('../reports/figures/04_risk_adj_revenue.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 3. Decision Intelligence — How Data Drives Business Outcomes

This section maps specific data capabilities to measurable business outcomes at Mercado Libre. The framework follows a **Capability → Decision → Outcome** chain.

In [ ]:
# ── 3.1  Decision Intelligence Framework ─────────────────────────────────────
di_framework = pd.DataFrame([
    {
        'Domain':      'Credit Underwriting',
        'Data_Input':  'Marketplace behavior, payment history, browse patterns',
        'Decision':    'Credit limit assignment, interest rate personalization',
        'Outcome':     'NPL improved from 8.4% (Q1 23) to 5.9% (Q2 24) while portfolio grew 61% YoY',
        'Signal':      'POSITIVE'
    },
    {
        'Domain':      'Fraud Detection',
        'Data_Input':  'Real-time transaction signals, device fingerprinting, network graph',
        'Decision':    'Block / allow / step-up authentication at payment moment',
        'Outcome':     'Chargeback rate consistently below 0.1% of TPV (industry avg ~0.3%)',
        'Signal':      'POSITIVE'
    },
    {
        'Domain':      'Logistics Routing',
        'Data_Input':  'Demand forecasts, warehouse inventory, carrier performance',
        'Decision':    'Fulfillment center allocation, last-mile carrier selection',
        'Outcome':     'Same-day/next-day delivery now >50% of shipments in BR & MX (Q4 2024)',
        'Signal':      'POSITIVE'
    },
    {
        'Domain':      'Seller Risk',
        'Data_Input':  'Seller ratings, return rates, complaint patterns, GMV concentration',
        'Decision':    'Seller tier access, cash advance eligibility, listing visibility',
        'Outcome':     'Counterfeit claims reduced YoY; Mercado Crédito merchant NPL < consumer NPL',
        'Signal':      'POSITIVE'
    },
    {
        'Domain':      'Ads & Monetization',
        'Data_Input':  'Search intent, browsing history, conversion probability',
        'Decision':    'Ad placement, bid floor pricing, budget pacing',
        'Outcome':     'Advertising revenue: $1.26B in FY2024 (+45% YoY) — fastest growing segment',
        'Signal':      'POSITIVE'
    },
    {
        'Domain':      'FX & Liquidity Risk',
        'Data_Input':  'Multi-currency flows across 18 markets, central bank policies',
        'Decision':    'Hedge ratios, stablecoin product design (Mercado Coin), reserve allocation',
        'Outcome':     'FX headwinds estimated at -$400M impact on FY2024 reported revenue',
        'Signal':      'RISK'
    },
])

print('=== DECISION INTELLIGENCE FRAMEWORK — MERCADO LIBRE 2024 ===')
for _, row in di_framework.iterrows():
    signal_icon = '✅' if row['Signal'] == 'POSITIVE' else '⚠️'
    print(f"\n{signal_icon} [{row['Domain']}]")
    print(f"   Input:    {row['Data_Input']}")
    print(f"   Decision: {row['Decision']}")
    print(f"   Outcome:  {row['Outcome']}")

In [ ]:
# ── 3.2  Decision Intelligence Score Map ─────────────────────────────────────
# Qualitative scoring (1-5) of data maturity across domains
# Based on: public disclosures, earnings call commentary, industry benchmarks

domains = ['Credit\nUnderwriting', 'Fraud\nDetection', 'Logistics\nRouting',
           'Seller\nRisk', 'Ads &\nMonetization', 'FX &\nLiquidity']

scores = {
    'Data Availability':  [5, 5, 4, 4, 5, 3],
    'Model Sophistication':[5, 5, 4, 3, 4, 3],
    'Decision Automation': [4, 5, 4, 3, 5, 2],
    'Outcome Measurability':[5, 4, 4, 3, 5, 3],
}

df_scores = pd.DataFrame(scores, index=domains)
composite = df_scores.mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
colors = [ACCENT_GREEN if v >= 4 else MELI_YELLOW if v >= 3 else ACCENT_RED for v in composite]
bars = ax.barh(domains, composite, color=colors, height=0.5)
ax.set_xlim(0, 5.5)
ax.set_title('Decision Intelligence Maturity Score by Domain (1–5)', fontsize=13)
ax.set_xlabel('Composite Score')
ax.axvline(x=4, color='#8b949e', linestyle='--', alpha=0.6, label='High maturity threshold')
for bar, val in zip(bars, composite):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f'{val:.2f}',
            va='center', color='#e6edf3', fontsize=10)
ax.legend(facecolor='#161b22', edgecolor='#30363d')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('../reports/figures/05_di_maturity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\nComposite DI Maturity Scores:')
print(composite.sort_values(ascending=False).round(2).to_string())

---
## 4. Key Findings & Analyst Notes

### Financial
- MELI crossed **$20B in annual revenue** for the first time in FY2024 (+43.5% YoY), with Commerce and Fintech nearly equal in size — a structural shift from marketplace-first to platform-first.
- Take rate expanded from ~33.9% (Q1 23) to ~45.2% (Q4 24), driven by advertising and credit penetration.

### Risk
- The credit portfolio grew **+75% YoY to $6.6B**, while NPL >90d stabilized in the 5.9–6.4% band — a signal that proprietary scoring models are absorbing volume growth without proportional deterioration.
- Coverage ratio ~97% provides a buffer, but **Q4 2024 uptick in NPL deserves monitoring** — could be early-cycle credit normalization or seasonal pattern.
- Provisions grew to **$2.4B in FY2024**, compressing risk-adjusted fintech margins. The RAR margin of ~76.7% (FY2024) is structurally lower than reported fintech revenue suggests.

### Decision Intelligence
- MELI's highest DI maturity is in **Fraud Detection and Ads Monetization** — both real-time, closed-loop systems with immediate measurable feedback.
- **FX & Liquidity** is the lowest-maturity domain: structural exposure to multi-currency risk in Argentina, Brazil, and Mexico limits data-driven automation. Stablecoin initiatives (Mercado Coin) are early-stage responses.
- The core DI moat is the **flywheel**: more buyers → more payment data → better credit scoring → lower NPL → more credit availability → more buyers.

---
*Next: `02_nubank_2024.ipynb` — Compare credit risk approach in a pure-play digital bank with no marketplace flywheel.*